# Semana 8: Derivadas Parciales y el Vector Gradiente
## Notebook de Ejercicios (NB2) - Dataset Boston Housing

### Propósito de la sesión
Aplicar el concepto de **gradiente** a un problema real de regresión con múltiples variables. Calcularemos manualmente el gradiente del MSE para todos los coeficientes, implementaremos el descenso de gradiente y visualizaremos cómo evoluciona la magnitud del gradiente a lo largo de las épocas.

### Objetivos de aprendizaje
*   Cargar y explorar el dataset **Boston Housing**.
*   Derivar la expresión del **gradiente del MSE** para regresión lineal múltiple.
*   Implementar el **descenso de gradiente** desde cero para todos los coeficientes.
*   Visualizar la **magnitud del gradiente** (norma) a lo largo del entrenamiento.
*   Analizar la convergencia del algoritmo y comparar con la solución analítica.

## Configuración Inicial

Importamos las librerías necesarias y cargamos el dataset Boston Housing. **Nota:** Boston Housing ya no está disponible en las versiones más recientes de sklearn, por lo que usaremos una alternativa o lo cargaremos desde un archivo local. En este caso, usaremos el dataset de California Housing como alternativa (similar en naturaleza) o intentaremos cargar Boston desde un repositorio.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)

# Cargamos el dataset (usamos California Housing como alternativa a Boston)
print("Cargando dataset California Housing...")
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = housing.feature_names

print(f"\n✅ Dataset cargado correctamente.")
print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"Características: {feature_names}")

### Exploración inicial de los datos

In [ ]:
# Creamos un DataFrame para visualizar
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print("Primeras 5 filas del dataset:")
display(df.head())

print("\nEstadísticas descriptivas:")
display(df.describe())

### Preprocesamiento

Dividimos en entrenamiento y prueba, y estandarizamos las características para que el gradiente converge mejor.

In [ ]:
# Dividimos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Estandarizamos las características (importante para gradiente descendente)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Añadimos columna de unos para el intercepto
X_train_b = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
X_test_b = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]

print(f"Shape X_train con intercepto: {X_train_b.shape}")
print(f"Shape X_test con intercepto: {X_test_b.shape}")

---
## Actividad 1: Calcular el gradiente del MSE para todos los coeficientes

### 1.1. Expresión del gradiente

Para regresión lineal múltiple con $n$ muestras y $d$ características, la función de costo MSE es:

$$J(\mathbf{w}) = \frac{1}{2n} \sum_{i=1}^n (\mathbf{x}_i^T \mathbf{w} - y_i)^2$$

El factor $\frac{1}{2}$ se añade por conveniencia para simplificar la derivada. El gradiente es:

$$\nabla J(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^n (\mathbf{x}_i^T \mathbf{w} - y_i) \mathbf{x}_i = \frac{1}{n} \mathbf{X}^T (\mathbf{X} \mathbf{w} - \mathbf{y})$$

### 1.2. Implementación del gradiente

In [ ]:
def mse_gradient(X, y, w):
    """
    Calcula el gradiente del MSE para regresión lineal.
    X: matriz de diseño (incluye columna de unos)
    y: vector objetivo
    w: vector de pesos (incluye intercepto)
    """
    n = len(y)
    y_pred = X @ w
    error = y_pred - y
    grad = (1/n) * X.T @ error
    return grad

# Inicializamos pesos (ceros)
w_init = np.zeros(X_train_b.shape[1])

# Calculamos el gradiente inicial
grad_inicial = mse_gradient(X_train_b, y_train, w_init)

print("🔷 Gradiente inicial (con pesos cero):")
print(f"Intercepto (gradiente): {grad_inicial[0]:.6f}")
for i, name in enumerate(feature_names):
    print(f"{name}: {grad_inicial[i+1]:.6f}")

---
## Actividad 2: Implementar descenso de gradiente y visualizar la magnitud

### 2.1. Algoritmo de descenso de gradiente

Actualizamos los pesos en la dirección opuesta al gradiente:

$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \nabla J(\mathbf{w}_t)$$

In [ ]:
def gradient_descent(X, y, w_init, lr=0.1, epochs=500, verbose=True):
    """
    Descenso de gradiente para regresión lineal.
    """
    w = w_init.copy()
    n = len(y)
    
    # Historia
    loss_history = []
    grad_norm_history = []
    w_history = [w.copy()]
    
    for epoch in range(epochs):
        # Calcular gradiente
        grad = mse_gradient(X, y, w)
        grad_norm = np.linalg.norm(grad)
        
        # Actualizar pesos
        w = w - lr * grad
        
        # Calcular pérdida
        y_pred = X @ w
        loss = (1/(2*n)) * np.sum((y_pred - y)**2)
        
        # Guardar historia
        loss_history.append(loss)
        grad_norm_history.append(grad_norm)
        w_history.append(w.copy())
        
        if verbose and epoch % 50 == 0:
            print(f"Epoch {epoch:3d}, Loss: {loss:.6f}, ||∇J||: {grad_norm:.6f}")
    
    return w, loss_history, grad_norm_history, w_history

# Ejecutamos gradiente descendente
lr = 0.5
epochs = 500
w_opt, loss_hist, grad_norm_hist, w_hist = gradient_descent(
    X_train_b, y_train, w_init, lr=lr, epochs=epochs
)

print(f"\n✅ Entrenamiento completado.")
print(f"Pérdida final: {loss_hist[-1]:.6f}")
print(f"Norma del gradiente final: {grad_norm_hist[-1]:.6f}")

### 2.2. Visualización de la pérdida y la magnitud del gradiente

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(loss_hist, 'b-', linewidth=2)
plt.xlabel('Época')
plt.ylabel('Pérdida (MSE/2)')
plt.title('Evolución de la función de pérdida')
plt.grid(True)
plt.yscale('log')

plt.subplot(1, 2, 2)
plt.plot(grad_norm_hist, 'r-', linewidth=2)
plt.xlabel('Época')
plt.ylabel('||∇J|| (norma del gradiente)')
plt.title('Evolución de la magnitud del gradiente')
plt.grid(True)
plt.yscale('log')

plt.suptitle(f'Descenso de Gradiente (lr={lr})', fontsize=14)
plt.tight_layout()
plt.show()

### 2.3. Interpretación

Observamos que tanto la pérdida como la magnitud del gradiente disminuyen a lo largo de las épocas. La norma del gradiente tiende a cero a medida que nos acercamos al mínimo, indicando que hemos alcanzado un punto crítico.

---
## Actividad 3: Comparación con la solución analítica y sklearn

### 3.1. Solución analítica (ecuación normal)

$$\mathbf{w}_{analitico} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

In [ ]:
# Ecuación normal
XTX = X_train_b.T @ X_train_b
XTy = X_train_b.T @ y_train
w_analitico = np.linalg.solve(XTX, XTy)

print("🔷 Coeficientes (ecuación normal):")
print(f"Intercepto: {w_analitico[0]:.6f}")
for i, name in enumerate(feature_names):
    print(f"{name}: {w_analitico[i+1]:.6f}")

In [ ]:
# 3.2. Solución con sklearn
model_sk = LinearRegression()
model_sk.fit(X_train_scaled, y_train)

print("🔶 Coeficientes (sklearn):")
print(f"Intercepto: {model_sk.intercept_:.6f}")
for i, name in enumerate(feature_names):
    print(f"{name}: {model_sk.coef_[i]:.6f}")

# 3.3. Comparación
print("\n📊 Comparación de coeficientes:")
print(f"{'Parámetro':<20} {'Gradiente':<15} {'Ecuación Normal':<15} {'sklearn':<15}")
print("-" * 65)

print(f"{'Intercepto':<20} {w_opt[0]:<15.6f} {w_analitico[0]:<15.6f} {model_sk.intercept_:<15.6f}")
for i, name in enumerate(feature_names):
    print(f"{name:<20} {w_opt[i+1]:<15.6f} {w_analitico[i+1]:<15.6f} {model_sk.coef_[i]:<15.6f}")

In [ ]:
# Evaluación en test
y_pred_gd = X_test_b @ w_opt
y_pred_analitico = X_test_b @ w_analitico
y_pred_sk = model_sk.predict(X_test_scaled)

mse_gd = mean_squared_error(y_test, y_pred_gd)
mse_analitico = mean_squared_error(y_test, y_pred_analitico)
mse_sk = mean_squared_error(y_test, y_pred_sk)

print(f"MSE en test - Gradiente: {mse_gd:.6f}")
print(f"MSE en test - Ecuación Normal: {mse_analitico:.6f}")
print(f"MSE en test - sklearn: {mse_sk:.6f}")

---
## Actividad 4: Análisis de la magnitud del gradiente en diferentes capas (concepto)

En una red neuronal profunda, los gradientes en diferentes capas pueden tener magnitudes muy diferentes. Esto se conoce como el problema del **gradiente desvaneciente** o **explosivo**. Aunque nuestro modelo es lineal, podemos simular este concepto.

In [ ]:
# Simulación conceptual: gradientes en diferentes "capas"
# Para un modelo lineal, todos los gradientes son de órdenes similares si las características están escaladas.
# Pero si no escalamos, algunas características pueden dominar.

# Comparación de magnitudes de gradientes por característica
grad_final = mse_gradient(X_train_b, y_train, w_opt)

plt.figure(figsize=(12, 6))
plt.bar(range(len(grad_final)), np.abs(grad_final))
plt.xticks(range(len(grad_final)), ['Intercepto'] + list(feature_names), rotation=45)
plt.xlabel('Parámetro')
plt.ylabel('|∂J/∂w| (magnitud del gradiente final)')
plt.title('Magnitud del gradiente para cada coeficiente (en el punto óptimo)')
plt.grid(True, alpha=0.3)
plt.show()

print("📌 En el óptimo, los gradientes deberían ser cercanos a cero. Aquí vemos que algunos aún son pequeños pero no cero debido a la convergencia.")

---
## Actividad 5: Experimentación con diferentes tasas de aprendizaje

La tasa de aprendizaje $\eta$ afecta la velocidad de convergencia y la estabilidad.

In [ ]:
learning_rates = [0.01, 0.1, 0.5, 1.0]
epochs = 200

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for lr in learning_rates:
    _, loss_hist_lr, _, _ = gradient_descent(
        X_train_b, y_train, w_init, lr=lr, epochs=epochs, verbose=False
    )
    plt.plot(loss_hist_lr, label=f'lr={lr}', linewidth=2)
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Efecto de la tasa de aprendizaje en la pérdida')
plt.legend()
plt.grid(True)
plt.yscale('log')

plt.subplot(1, 2, 2)
for lr in learning_rates:
    _, _, grad_norm_hist_lr, _ = gradient_descent(
        X_train_b, y_train, w_init, lr=lr, epochs=epochs, verbose=False
    )
    plt.plot(grad_norm_hist_lr, label=f'lr={lr}', linewidth=2)
plt.xlabel('Época')
plt.ylabel('||∇J||')
plt.title('Efecto en la magnitud del gradiente')
plt.legend()
plt.grid(True)
plt.yscale('log')

plt.suptitle('Comparación de tasas de aprendizaje')
plt.tight_layout()
plt.show()

---
## Ejercicios para el Estudiante

1.  **Diferente inicialización:** Prueba inicializar los pesos con valores aleatorios en lugar de ceros. ¿Cambia la velocidad de convergencia?

2.  **Mini-batch Gradient Descent:** Modifica el algoritmo para usar mini-batches (por ejemplo, batches de 32 muestras) en lugar de todo el dataset. Compara la evolución de la magnitud del gradiente.

3.  **Regularización L2:** Añade un término de regularización L2 a la función de costo: $J_{ridge} = MSE + \lambda \|w\|^2$. Deriva el nuevo gradiente e implementa **Ridge Regression** con descenso de gradiente.

4.  **Características no estandarizadas:** Ejecuta el mismo algoritmo **sin** estandarizar las características. ¿Qué observas en la magnitud de los gradientes? ¿Converge más lento o más rápido?

5.  **Reflexión:** ¿Por qué es importante monitorear la magnitud del gradiente durante el entrenamiento? ¿Qué indicaría una magnitud que no disminuye o que aumenta?

---
## Conclusiones de la Sesión

*   Hemos **calculado manualmente el gradiente del MSE** para regresión lineal múltiple, obteniendo la expresión matricial $\nabla J = \frac{1}{n} \mathbf{X}^T (\mathbf{X} \mathbf{w} - \mathbf{y})$.
*   Implementamos el **descenso de gradiente** para entrenar un modelo con el dataset California Housing, actualizando todos los coeficientes simultáneamente.
*   Visualizamos la **evolución de la magnitud del gradiente** a lo largo de las épocas, observando cómo tiende a cero al acercarse al mínimo.
*   Comparamos nuestros resultados con la **ecuación normal** y **sklearn**, verificando que todos los métodos convergen a la misma solución.
*   Experimentamos con diferentes **tasas de aprendizaje**, observando su impacto en la convergencia.
*   Este ejercicio consolida la conexión entre el **vector gradiente** y el entrenamiento de modelos, mostrando cómo guía la actualización de los parámetros.

¡Ahora entiendes cómo el gradiente dirige el aprendizaje en modelos con múltiples parámetros!